# TCSS590-SP26 HW1: Supervised Learning of Behaviors

Starter-only Google Colab companion for **TCSS590-SP26: Theory and Algorithms of Reinforcement Learning, Homework 1**.

This notebook sets up the runtime, loads the starter repository, downloads expert data if needed, and provides run cells for behavior cloning, DAgger, and autoregressive-policy experiments. It intentionally does **not** fill in the assignment TODOs or include solution code.


## Runtime setup

Use **Runtime -> Change runtime type -> T4 GPU** when available. CPU can run smoke tests, but the full experiments may be slow. Run cells in order. If Colab asks you to restart after installing packages, restart and rerun from the install cell.


In [ ]:
import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())


## Install dependencies

The repository uses `environment.yml` for conda. In Colab, this cell installs the corresponding pip packages. Colab usually already includes PyTorch, so PyTorch is installed only if missing.


In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    'numpy==1.26.4',
    'matplotlib==3.8.4',
    'gymnasium==0.29.1',
    'gymnasium-robotics==1.2.4',
    'mujoco==2.3.7',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

if importlib.util.find_spec('torch') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.3.1', 'torchvision==0.18.1', 'torchaudio==2.3.1'
    ])

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import mujoco
import torch

print('numpy:', np.__version__)
print('gymnasium:', gym.__version__)
print('mujoco:', mujoco.__version__)
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Load the starter project

After the public GitHub repository is created, leave `USE_GITHUB = True`. Before publication, set `USE_GITHUB = False` and upload the starter zip through the Colab file picker.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

USE_GITHUB = True
REPO_URL = 'https://github.com/liuxt/tcss590-rl-coding-assignment-1.git'
REPO_DIR = Path('/content/tcss590-rl-coding-assignment-1')

if USE_GITHUB:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    if not zip_names:
        raise RuntimeError('Upload a .zip file containing the starter repository.')
    zip_path = Path('/content') / zip_names[0]
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content')

    if not REPO_DIR.exists():
        candidates = [
            Path('/content/tcss590-rl-coding-assignment-1-starter'),
            *[p for p in Path('/content').iterdir() if p.is_dir() and p.name.startswith('tcss590-rl-coding-assignment-1')],
        ]
        candidates = [p for p in candidates if p.exists()]
        if not candidates:
            raise FileNotFoundError('Could not find the extracted starter repository folder.')
        shutil.move(str(candidates[0]), str(REPO_DIR))

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())


## Verify starter files

This checks the project structure only. It does not check whether the TODOs have been completed.


In [ ]:
from pathlib import Path

required_files = [
    'main.py', 'bc.py', 'dagger.py', 'utils.py', 'evaluate.py',
    'environment.yml', 'policy.py', 'scripts/download_data.sh'
]
optional_files = ['run_experiments.py', 'TCSS590_SP26_HW1.ipynb']
missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError('Missing required starter files: ' + ', '.join(missing))
print('Required starter files are present.')
for f in optional_files:
    print(f'{f}:', 'present' if Path(f).exists() else 'not present')


## Download expert data

The public starter repository should not commit the large `.pkl` files. This cell downloads them into `data/` if they are missing.


In [ ]:
from pathlib import Path
import subprocess

required_data = [
    Path('data/reacher_expert_data.pkl'),
    Path('data/reacher_expert_policy.pkl'),
    Path('data/pointmaze_expert_data.pkl'),
]
missing_data = [p for p in required_data if not p.exists()]
if missing_data:
    print('Missing data files:', [str(p) for p in missing_data])
    subprocess.check_call(['bash', 'scripts/download_data.sh'])
else:
    print('Expert data files are already present.')

for p in required_data:
    if not p.exists():
        raise FileNotFoundError(p)
    print(f'{p}: {p.stat().st_size / (1024 * 1024):.2f} MB')


## Quick import smoke test

This verifies that the local modules and environments import correctly. It does not complete any assignment TODOs.


In [ ]:
import torch
import gymnasium as gym
from reach_goal.envs.pointmaze_env import PointMazeEnv
from utils import get_expert_data, PolicyGaussian, PolicyAutoRegressiveModel

reacher = gym.make('Reacher-v4')
pointmaze = PointMazeEnv(render_mode='rgb_array')
print('Reacher observation/action shapes:', reacher.observation_space.shape, reacher.action_space.shape)
print('PointMaze observation/action shapes:', pointmaze.observation_space.shape, pointmaze.action_space.shape)
reacher.close(); pointmaze.close()
print('Imports and environment construction succeeded.')


## Complete the assignment TODOs

Before running the training cells, edit these files in Colab or locally:

- `bc.py`: implement behavior cloning.
- `dagger.py`: implement DAgger.
- `utils.py`: implement `PolicyAutoRegressiveModel`.

Colab's file browser can open and edit these files directly. Save your edits before running experiments.


## Behavior cloning with Gaussian policy

Run these after implementing the behavior cloning TODOs in `bc.py`.


In [ ]:
!mkdir -p logs
!python main.py --env reacher --train behavior_cloning --policy gaussian | tee logs/bc_reacher_gaussian.log


In [ ]:
!mkdir -p logs
!python main.py --env pointmaze --train behavior_cloning --policy gaussian | tee logs/bc_pointmaze_gaussian.log


## DAgger with Gaussian policy

Run these after implementing the DAgger TODOs in `dagger.py`.


In [ ]:
!mkdir -p logs
!python main.py --env reacher --train dagger --policy gaussian | tee logs/dagger_reacher_gaussian.log


In [ ]:
!mkdir -p logs
!python main.py --env pointmaze --train dagger --policy gaussian | tee logs/dagger_pointmaze_gaussian.log


## Autoregressive policies on PointMaze

Run these after implementing `PolicyAutoRegressiveModel` in `utils.py`.


In [ ]:
!mkdir -p logs
!python main.py --env pointmaze --train behavior_cloning --policy autoregressive | tee logs/bc_pointmaze_autoregressive.log


In [ ]:
!mkdir -p logs
!python main.py --env pointmaze --train dagger --policy autoregressive | tee logs/dagger_pointmaze_autoregressive.log


## Optional: run the helper sweep script

The starter repository includes `run_experiments.py`, which can run default experiments and save plots/metrics under `results/`. Use this only after completing the relevant TODOs.


In [ ]:
# Example: run all default Gaussian experiments.
# !python run_experiments.py

# Example: run only behavior-cloning experiments.
# !python run_experiments.py --only behavior_cloning


## Plot losses from log files

The helper below parses lines containing `loss:` from saved logs and plots the resulting sequence.


In [ ]:
import re
from pathlib import Path
import matplotlib.pyplot as plt

LOSS_RE = re.compile(r'loss:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')

def parse_losses(log_path):
    log_path = Path(log_path)
    if not log_path.exists():
        raise FileNotFoundError(log_path)
    losses = []
    for line in log_path.read_text(errors='ignore').splitlines():
        match = LOSS_RE.search(line)
        if match:
            losses.append(float(match.group(1)))
    return losses

def plot_losses(log_path, title=None):
    losses = parse_losses(log_path)
    if not losses:
        raise ValueError(f'No loss entries found in {log_path}')
    plt.figure(figsize=(7, 4))
    plt.plot(range(len(losses)), losses)
    plt.xlabel('Logged training step')
    plt.ylabel('Loss')
    plt.title(title or str(log_path))
    plt.grid(True, alpha=0.3)
    plt.show()
    print(f'Parsed {len(losses)} losses from {log_path}')

# plot_losses('logs/bc_reacher_gaussian.log', 'BC Gaussian - Reacher')


In [ ]:
# Uncomment any plot after the corresponding run cell has completed.
# plot_losses('logs/bc_reacher_gaussian.log', 'BC Gaussian - Reacher')
# plot_losses('logs/bc_pointmaze_gaussian.log', 'BC Gaussian - PointMaze')
# plot_losses('logs/dagger_reacher_gaussian.log', 'DAgger Gaussian - Reacher')
# plot_losses('logs/dagger_pointmaze_gaussian.log', 'DAgger Gaussian - PointMaze')
# plot_losses('logs/bc_pointmaze_autoregressive.log', 'BC Autoregressive - PointMaze')
# plot_losses('logs/dagger_pointmaze_autoregressive.log', 'DAgger Autoregressive - PointMaze')


## Optional: package code for Canvas submission

This creates a zip of your working directory. By default, it excludes expert data files and learned checkpoints to keep the submission small.


In [ ]:
from pathlib import Path
import zipfile

submission_name = 'tcss590_hw1_submission.zip'
exclude_suffixes = {'.pyc', '.pkl', '.pth', '.pt'}
exclude_dirs = {'.git', '__pycache__', '.ipynb_checkpoints'}

with zipfile.ZipFile(submission_name, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in Path('.').rglob('*'):
        if path.is_dir():
            continue
        parts = set(path.parts)
        if parts & exclude_dirs:
            continue
        if path.suffix in exclude_suffixes:
            continue
        if path.name == submission_name:
            continue
        zf.write(path, path.as_posix())

print(f'Created {submission_name}')
try:
    from google.colab import files
    files.download(submission_name)
except Exception:
    print('Download manually from the Colab file browser.')
